# V2_5 (version 4) - LULC map figures (Figs 3, 4, 10) and methodology flowchart (Fig. 2)
Run on **Colab (CPU)** after `V2_1`, `V2_2` and `V2_3` have finished. It reads the GeoTIFFs listed below and writes print-size figures (6.3 in = 16 cm wide, Times-style 10-13 pt text as in the edited v2 notebook; maps are outlined with your study-area shapefile, each map sits in a padded frame with left/bottom latitude-longitude ticks, an alternating scale bar and a north arrow; PNG 300 dpi, vector PDF, 600 dpi TIFF) to `revised_outputs_v2_REPORT/manuscript_maps/`.

| Manuscript figure | GeoTIFFs used (all in `PATH`, i.e. `/content/drive/MyDrive/Obj2_LULC_GWP/`) |
|---|---|
| **Fig. 3** observed LULC 2003, 2013, 2023 | `LULC_2003.tif`, `LULC_2013.tif`, `LULC_2023.tif` |
| **Fig. 4** hindcast of 2023 | observed `LULC_2023.tif`; persistence = `LULC_2013.tif`; model maps `revised_outputs_v2_anomaly/{CA_ANN_Markov, CNN_Markov, CNN_argmax}_hind2023_seed{0-4}.tif` (majority vote over the five seeds, the same ensemble used for the bootstrap in Table 4) |
| **Fig. 10** projection to 2033 | `revised_outputs_v2_anomaly/CNN_Markov_fore2033_{T2, Mean, T1}_seed{0-4}.tif` (majority vote over seeds) and `LULC_2023.tif` for the change maps |

Do **not** use the `*_global`, `*_noclim`, `*_stateonly` folders, the shield variants, or single seeds for the main-text maps. Fig. 2 (flowchart) needs no data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============ 0. SETUP ============
import os, math, warnings, subprocess
import numpy as np
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Rectangle, FancyBboxPatch, FancyArrowPatch, Patch
import matplotlib.font_manager as fm
warnings.filterwarnings('ignore')
try:
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-liberation'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)   # Times-compatible font on Colab
except Exception: pass
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass

PATH  = globals().get('PATH', '/content/drive/MyDrive/Obj2_LULC_GWP/')
FOLD  = os.path.join(PATH, 'revised_outputs_v2_anomaly')          # primary configuration
SEEDS = [0, 1, 2, 3, 4]
OUT   = globals().get('OUT', os.path.join(PATH, 'revised_outputs_v2_REPORT', 'manuscript_maps')); os.makedirs(OUT, exist_ok=True)
CLASS_VALUES = [1, 2, 3, 4, 5]; CLASS_NAMES = ['Water', 'Forest', 'Agriculture', 'Settlement', 'Fallow']; K = 5
CLS_COL = ['#2C7BB6', '#1A7F37', '#E8DE8A', '#D7301F', '#B8895A']          # water, forest, agriculture, settlement, fallow (colour-blind distinguishable)

for _f in fm.findSystemFonts():
    if 'iberation' in _f and 'Serif' in _f: fm.fontManager.addfont(_f)
SERIF = [f for f in ['Times New Roman', 'Liberation Serif', 'Tinos', 'TeX Gyre Termes', 'DejaVu Serif'] if f in {x.name for x in fm.fontManager.ttflist}]
mpl.rcParams.update({'font.family': 'serif', 'font.serif': SERIF, 'mathtext.fontset': 'stix', 'font.size': 11.5, 'axes.titlesize': 13.0, 'legend.fontsize': 11.5,
                     'axes.linewidth': 0.8, 'pdf.fonttype': 42, 'savefig.dpi': 300})
W = 6.3
def savefig(fig, name):
    fig.savefig(os.path.join(OUT, f'{name}.png'), dpi=300, bbox_inches='tight', pad_inches=0.03)
    fig.savefig(os.path.join(OUT, f'{name}.pdf'), bbox_inches='tight', pad_inches=0.03)
    fig.savefig(os.path.join(OUT, f'{name}.tiff'), dpi=600, bbox_inches='tight', pad_inches=0.03, pil_kwargs={'compression': 'tiff_lzw'})
    plt.show(); print('saved', name)
print('font:', SERIF[0])

## 1. Methodology flowchart (Fig. 2) - text is 10 pt; box heights follow the text, so edit the strings and the layout re-fits

In [ ]:
def flowchart(path_stem='Fig2_methodology_flowchart', outdir='.', FS=10.0):
    """Methodology flowchart. Box heights are computed from the number of text lines, so boxes stay tight when FS changes."""
    import os
    from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
    W = 6.3; LH = FS * 1.22 / 72; PV = 0.045; LBL = FS * 1.35 / 72; GB = 0.20            # width, line height, box padding, band-label height, gap between bands (inches)
    fig = plt.figure(figsize=(W, 10.5)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(10.5, 0); ax.axis('off')
    C = {'data': '#DCEBF7', 'prep': '#E4F2E1', 'model': '#FCE9D2', 'stage': '#EADCF2', 'eval': '#F8E1E1', 'out': '#EFEFEF', 'edge': '#3A3A3A'}
    X0, IW = 0.12, 6.06
    def cols(n, gap):
        w = (IW - (n - 1) * gap) / n; return [(X0 + i * (w + gap), w) for i in range(n)]
    def bh(t): return (t.count('\n') + 1) * LH + 2 * PV
    def box(x, y, w, h, t, fc='white', bold=False):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0,rounding_size=0.04', fc=fc, ec=C['edge'], lw=0.8, zorder=2))
        if bold:
            l = t.split('\n'); ax.text(x + w / 2, y + PV, l[0], ha='center', va='top', fontsize=FS, fontweight='bold', zorder=3)
            ax.text(x + w / 2, y + PV + LH, '\n'.join(l[1:]), ha='center', va='top', fontsize=FS, linespacing=1.15, zorder=3)
        else:
            ax.text(x + w / 2, y + h / 2, t, ha='center', va='center', fontsize=FS, linespacing=1.15, zorder=3)
    def arrow(x0, y0, x1, y1):
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle='-|>', mutation_scale=9, lw=0.9, color=C['edge'], zorder=4, shrinkA=0, shrinkB=0))
    def line(xs, ys): ax.plot(xs, ys, color=C['edge'], lw=0.9, zorder=4, solid_capstyle='butt')
    def band(y0, y1, color, label):
        ax.add_patch(FancyBboxPatch((0.05, y0), W - 0.10, y1 - y0, boxstyle='round,pad=0,rounding_size=0.05', fc=color, ec='none', alpha=0.6, zorder=0))
        ax.text(0.14, y0 + 0.03, label, fontsize=FS + 0.5, fontweight='bold', va='top', ha='left', zorder=3)
    def row(y, specs, bolds=None):
        h = max(bh(t) for t, _ in specs); out = []
        for (t, fc), (x, w), b in zip(specs, row.cols, bolds or [False] * len(specs)): box(x, y, w, h, t, fc, b); out.append((x, w))
        return h, out

    # ---------------- 1 Data
    y = 0.03; b0 = y; y += 0.03 + LBL + 0.03
    row.cols = cols(3, 0.15)
    h, D = row(y, [('Landsat 7 ETM+ (2003)\nLandsat 8 OLI (2013)\nLandsat 9 OLI-2 (2023)\npre-monsoon scenes', 'white'),
                   ('Classified LULC maps\n(five classes) with\naccuracy assessment\n(500 points per map)', 'white'),
                   ('Covariates: NDVI, MNDWI,\nNDMI, LST, water distance,\nCHIRPS rainfall; SRTM DEM,\nslope, DD, Ld, TWI', 'white')])
    ybD = y + h; arrow(D[0][0] + D[0][1], y + h / 2, D[1][0], y + h / 2)
    y = ybD + 0.05; band(b0, y, C['data'], '1  Data'); bandD = y
    # ---------------- 2 Preparation
    ymid = bandD + GB / 2; y = bandD + GB; b0 = y; y += 0.03 + LBL + 0.03
    h, P = row(y, [('Reproject to 2023 grid\n(30 m; 2,399 × 2,424)', 'white'), ('Within-year\nstandardisation (Eq. 1)', 'white'), ('Spatial blocks of 64 px:\n70 / 15 / 15 split', 'white')])
    xin = P[0][0] + 0.78 * P[0][1]; c1 = D[1][0] + D[1][1] / 2; c2 = D[2][0] + D[2][1] / 2
    line([c1, c1], [ybD, ymid]); line([c2, c2], [ybD, ymid]); line([xin, c2], [ymid, ymid]); arrow(xin, ymid, xin, y)
    arrow(P[0][0] + P[0][1], y + h / 2, P[1][0], y + h / 2); arrow(P[1][0] + P[1][1], y + h / 2, P[2][0], y + h / 2)
    ybP = y + h; y = ybP + 0.05; band(b0, y, C['prep'], '2  Preparation'); bandP = y
    # ---------------- 3 Models
    ymid = bandP + GB / 2; y = bandP + GB; b0 = y; y += 0.03 + LBL + 0.03
    row.cols = cols(2, 0.15)
    h, M = row(y, [('Suitability learners (p$_k$)\nANN (64–32 units) | patch CNN (5 × 5)\nclass-weighted loss (Eq. 2)', 'white'),
                   ('Markov demand (Eqs. 3–4)\n+ CA contiguity (Eq. 5)\n+ competitive allocation (Eq. 6)', 'white')])
    ybM = y + h; m0 = M[0][0] + M[0][1] / 2; m1 = M[1][0] + M[1][1] / 2
    p1 = P[1][0] + P[1][1] / 2; line([p1, p1], [ybP, ymid]); line([m0, p1], [ymid, ymid]); arrow(m0, ymid, m0, y)
    arrow(M[0][0] + M[0][1], y + h / 2, M[1][0], y + h / 2)
    yj = ybM + 0.12; line([m0, m0], [ybM, yj]); line([m1, m1], [ybM, yj]); line([m0, m1], [yj, yj])
    xm = (m0 + m1) / 2; y2 = yj + 0.28; line([xm, xm], [yj, y2])
    ax.text(0.14, (yj + y2) / 2, 'Configurations compared', fontsize=FS - 0.3, style='italic', va='center', ha='left', zorder=3)
    y = y2 + 0.11; row.cols = cols(4, 0.10)
    h, Cf = row(y, [('CA-ANN-Markov\n(ANN + demand)', C['model']), ('CNN-Markov\n(CNN + demand)', C['model']), ('CNN-argmax\n(no demand control)', C['model']), ('Persistence\n(no-change baseline)', 'white')])
    cc = [x + w / 2 for x, w in Cf]; line([cc[0], cc[3]], [y2, y2])
    for c in cc: arrow(c, y2, c, y)
    ybC = y + h; y = ybC + 0.05; band(b0, y, C['model'], '3  Models'); bandM = y
    # ---------------- 4 Two stages
    ymid = bandM + GB / 2; y = bandM + GB; b0 = y; y += 0.03 + LBL + 0.03
    row.cols = cols(2, 0.15)
    h, S = row(y, [('Stage V: out-of-time validation\nCalibrate on T1 (2003 → 2013)\nPredict 2023 from 2013\nCompare with observed 2023\n(held-out blocks and all blocks)', 'white'),
                   ('Stage F: projection to 2033\nCalibrate on T2 (2013 → 2023)\nPredict 2033 from 2023\nRate scenarios: T1, mean, T2\n(five seeds each)', 'white')], bolds=[True, True])
    sv = S[0][0] + S[0][1] / 2; sf = S[1][0] + S[1][1] / 2
    for c in cc: line([c, c], [ybC, ymid])
    line([cc[0], cc[3]], [ymid, ymid]); arrow(sv, ymid, sv, y); arrow(sf, ymid, sf, y)
    ybS = y + h; y = ybS + 0.05; band(b0, y, C['stage'], '4  Two stages'); bandS = y
    # ---------------- 5 Evaluation
    y = bandS + GB; b0 = y; y += 0.03 + LBL + 0.03
    t0 = 'OA, kappa, F1; FoM, quantity and\nallocation disagreement (Eqs. 7–8)\nsteep-slope water (Eq. 9)\nblock bootstrap (Eq. 10); ablations;\nFallow → Forest precision (Eq. 11)'
    t1 = 'Class-area change by scenario (km²)\nseed spread versus scenario spread (η²)\nyear-label check on unconstrained model\nsteep-slope water check'
    h, E = row(y, [(t0, 'white'), (t1, 'white')])
    arrow(sv, ybS, sv, y); arrow(sf, ybS, sf, y)
    ybE = y + h; y = ybE + 0.05; band(b0, y, C['eval'], '5  Evaluation'); bandE = y
    # ---------------- reporting
    y = bandE + GB; row.cols = cols(1, 0)
    rep = 'Reporting: persistence-referenced skill, scenario ranges, and\nindependent check of the Fallow → Forest flow'
    h, _ = row(y, [(rep, C['out'])])
    arrow(E[0][0] + E[0][1] / 2, ybE, E[0][0] + E[0][1] / 2, y); arrow(E[1][0] + E[1][1] / 2, ybE, E[1][0] + E[1][1] / 2, y)
    total = y + h + 0.03
    fig.set_size_inches(W, total); ax.set_ylim(total, 0)
    for ext, kw in [('png', dict(dpi=300)), ('pdf', {}), ('tiff', dict(dpi=600, pil_kwargs={'compression': 'tiff_lzw'}))]:
        fig.savefig(os.path.join(outdir, f'{path_stem}.{ext}'), **kw)
    print(f'flowchart: {W:.2f} x {total:.2f} in, text {FS} pt')
    return fig

flowchart(outdir=OUT); plt.show()

## 2. Load the GeoTIFFs (also reads the georeferencing used for the coordinate ticks)

In [ ]:
import rasterio
def read_idx(p):
    with rasterio.open(p) as s: a = s.read(1); res = abs(s.res[0])
    o = np.full(a.shape, -1, np.int8)
    for k, v in enumerate(CLASS_VALUES): o[a == v] = k
    return o, res
LULC, PIX = {}, 30.0
for y in ['2003', '2013', '2023']:
    LULC[y], PIX = read_idx(os.path.join(PATH, f'LULC_{y}.tif'))
VALID = np.all([LULC[y] >= 0 for y in LULC], axis=0); H, W_PX = VALID.shape
with rasterio.open(os.path.join(PATH, 'LULC_2023.tif')) as _s: _TF, _CRS = _s.transform, _s.crs
print('grid', H, 'x', W_PX, '| pixel size', PIX, 'm | CRS', _CRS)

SEED_CHG = {}      # mean over seeds of the % of valid pixels that change from 2023 (the number reported in Table 7)
def vote(maps):
    st = np.stack(maps); return np.where(VALID, np.stack([(st == k).sum(0) for k in range(K)]).argmax(0), -1).astype(np.int8)
def ens(tag):
    fs = [os.path.join(FOLD, f'{tag}_seed{s}.tif') for s in SEEDS]; fs = [f for f in fs if os.path.exists(f)]
    print(f'  {tag}: {len(fs)} seeds')
    if not fs: return None
    maps = [read_idx(f)[0] for f in fs]
    SEED_CHG[tag] = float(np.mean([100 * ((m != LULC['2023']) & VALID).sum() / VALID.sum() for m in maps]))
    return vote(maps)
HIND = {m: ens(f'{m}_hind2023') for m in ['CA_ANN_Markov', 'CNN_Markov', 'CNN_argmax']}
FORE = {sc: ens(f'CNN_Markov_fore2033_{sc}') for sc in ['T2', 'Mean', 'T1']}
SEED_CHG = {k.replace('CNN_Markov_fore2033_', ''): v for k, v in SEED_CHG.items() if 'fore2033' in k}
print('per-seed mean change (%):', {k: round(v, 2) for k, v in SEED_CHG.items()})

## 3. Map figures: Fig. 3 (observed LULC), Fig. 4 (hindcast), Fig. 10 (2033 projection)
This cell reproduces the look of your edited v2 notebook (font sizes, spacing, 5+5+10+10+10 km scale bar, red dashed zoom box, shapefile outline) with the fixes listed in the notes below. It looks for `Khatra_LULC_aligned_dissolved.shp` (then `khatra_boundary.shp`, then any boundary-like shapefile) in `PATH`; set `BOUNDARY_PATH` at the top of the cell to force a file. Switches at the top of the cell: `PANEL_TITLES` ('letters' = only (a), (b), ... above each frame, the default; 'full' = the old descriptive titles), `CROP_TO_BOUNDARY`, `MASK_OUTSIDE`, `ZOOM_IGNORE_WATER`, `SCALEBAR_FS`.
* **Panel titles (v5).** Descriptive titles are removed; each frame carries only its panel letter. The description, and the Fig. 10 percentages that used to sit in the lower titles, now belong in the caption. The cell prints ready-to-paste caption text with the numbers from your run. Set `PANEL_TITLES = 'full'` to restore the old titles. Fig. 10 no longer needs two-line titles, so its top band is slimmer.

In [ ]:
# ---------------- layout constants (all in inches, so the printed size is exact); values follow your edited notebook ----------------
PADX, PADT, PADB = 0.07, 0.07, 0.42          # white space between map and frame: sides, top, bottom (bottom band carries scale bar + north arrow)
SEP, RM = 0.35, 0.04                          # space left of each frame for the latitude labels; right margin
TT1, TT2, TL = 0.35, 0.50, 0.30               # title height (1 or 2 lines); space under a frame for longitude labels
FULL_TICKS = (15, 5)                          # (labelled interval, minor-tick interval) in arc-minutes for whole-area panels
ZOOM_TICKS = (5, 1)                           # the 15.6 km zoom window is only ~9' wide, so 15' ticks would not fall inside it
BAR_FULL = (10, 5)                            # (primary division km, secondary division km): bar = 5 + 5 + 10 + 10 + 10 = 40 km
BAR_ZOOM = (2, 1)                             # zoom window: 1 + 1 + 2 + 2 + 2 = 8 km
TICK_MAJOR, TICK_MINOR, TICK_FS = 4.0, 2.0, 11.5    # tick length (pt) and tick-label size
SCALEBAR_FS = 8.5                             # scale-bar numbers and unit; the "5" and "10" labels are only ~0.11 in apart, so 11.5 pt would make them touch
TITLE_FS, LEGEND_FS, LEGEND_TITLE_FS = 13.0, 11.5, 12.5
PANEL_TITLES = 'letters'                         # 'letters' = only (a), (b), (c) ... above each frame (description goes in the figure caption); 'full' = the previous descriptive titles
def TT(letter, full): return f'({letter})' if PANEL_TITLES == 'letters' else f'({letter}) {full}'
PCT_LOG, ZOOM_KM = {}, [None]                    # filled while drawing; used to print caption text with the actual numbers
OUTLINE_LW = 0.5                              # study-area outline (pt)
CROP_TO_BOUNDARY = False                      # True: crop each frame to the boundary's bounding box (+ MARGIN_PX); False: whole raster, as in your notebook
MASK_OUTSIDE = False                          # True: also paint pixels outside the polygon white (your rasters are already NoData there)
ZOOM_IGNORE_WATER = False                     # True: choose the Fig. 4 zoom window by non-water change only (the default window lands on the reservoir)

# ---------------- study-area boundary (vector file from your Drive) ----------------
BOUNDARY_PATH = None          # None = look for Khatra_LULC_aligned_dissolved.shp etc., then search PATH; or e.g. '/content/drive/MyDrive/Obj2_LULC_GWP/Khatra_boundary.shp'; None = search PATH for a likely file
INSIDE, BOUND, FULLWIN = None, [], None
MARGIN_PX = 20                # extra pixels kept around the boundary when the frame is cropped to it

def find_boundary(path):
    import glob
    if BOUNDARY_PATH: return BOUNDARY_PATH
    for nm in ('Khatra_LULC_aligned_dissolved.shp', 'khatra_boundary.shp', 'Khatra_boundary.shp'):          # the names your notebook looked for
        if os.path.exists(os.path.join(path, nm)): return os.path.join(path, nm)
    cands = [f for ext in ('shp', 'geojson', 'gpkg') for f in glob.glob(os.path.join(path, f'**/*.{ext}'), recursive=True)]
    pref = [f for f in cands if any(k in os.path.basename(f).lower() for k in ('khatra', 'study', 'boundary', 'aoi', 'subdiv'))]
    print('boundary candidates:', [os.path.basename(f) for f in cands]); return (pref or [None])[0]

def load_boundary(path, tf, crs, shape):
    """Read the study-area polygon(s), reproject to the raster CRS, rasterise the inside mask, and return outline coordinates in pixel units."""
    import geopandas as gpd, rasterio.features as rf
    g = gpd.read_file(path)
    if g.crs is not None and crs is not None: g = g.to_crs(crs)
    geom = g.unary_union; inv = ~tf; polys = list(geom.geoms) if hasattr(geom, 'geoms') else [geom]
    inside, lines = None, []
    if all(p.geom_type == 'Polygon' for p in polys):
        inside = rf.rasterize([(geom, 1)], out_shape=shape, transform=tf, fill=0, dtype='uint8').astype(bool)
        rings = [r for p in polys for r in [p.exterior] + list(p.interiors)]
    else: rings = polys                                                   # line layer: outline only, no mask
    for r in rings:
        xy = np.asarray(r.coords); c, rr = inv * (xy[:, 0], xy[:, 1]); lines.append((np.asarray(c) - 0.5, np.asarray(rr) - 0.5))   # imshow pixel-centre convention
    return inside, lines

def set_boundary(inside, lines, shape):
    """Outline is always drawn. Masking and cropping only if MASK_OUTSIDE / CROP_TO_BOUNDARY are True."""
    global INSIDE, BOUND, FULLWIN
    BOUND = lines; INSIDE = inside if (MASK_OUTSIDE and inside is not None) else None
    if CROP_TO_BOUNDARY and inside is not None and inside.any():
        rr, cc = np.where(inside.any(1))[0], np.where(inside.any(0))[0]; Hh, Ww = shape
        r0, r1 = max(rr.min() - MARGIN_PX, 0), min(rr.max() + MARGIN_PX + 1, Hh); c0, c1 = max(cc.min() - MARGIN_PX, 0), min(cc.max() + MARGIN_PX + 1, Ww)
        FULLWIN = (int(r0), int(c0), int(r1 - r0), int(c1 - c0))
    else: FULLWIN = (0, 0) + tuple(shape)
    print('frame window (row0, col0, rows, cols):', FULLWIN, '| outline segments:', len(BOUND), '| mask outside polygon:', INSIDE is not None)

def fullwin(): return FULLWIN or (0, 0, H, W_PX)
def crop(a, win):
    r0, c0, h, w = win; return a[r0:r0 + h, c0:c0 + w]
def outside_mask(win): return None if INSIDE is None else ~crop(INSIDE, win)

# ---------------- georeferencing -> lon/lat for the frame ticks ----------------
def _utm_inverse(x, y, zone, north=True):
    """WGS84 UTM -> lon, lat (degrees); Krueger series, accurate to millimetres. Used only if pyproj is unavailable."""
    a, f = 6378137.0, 1 / 298.257223563; n = f / (2 - f); k0 = 0.9996
    A = a / (1 + n) * (1 + n**2 / 4 + n**4 / 64)
    b = [n / 2 - 2 * n**2 / 3 + 37 * n**3 / 96, n**2 / 48 + n**3 / 15, 17 * n**3 / 480]
    d = [2 * n - 2 * n**2 / 3 - 2 * n**3, 7 * n**2 / 3 - 8 * n**3 / 5, 56 * n**3 / 15]
    xi = (y - (0 if north else 1e7)) / (k0 * A); eta = (x - 5e5) / (k0 * A)
    xi_p = xi - sum(b[j] * np.sin(2 * (j + 1) * xi) * np.cosh(2 * (j + 1) * eta) for j in range(3))
    eta_p = eta - sum(b[j] * np.cos(2 * (j + 1) * xi) * np.sinh(2 * (j + 1) * eta) for j in range(3))
    chi = np.arcsin(np.sin(xi_p) / np.cosh(eta_p))
    lat = chi + sum(d[j] * np.sin(2 * (j + 1) * chi) for j in range(3))
    lon = np.radians((zone - 1) * 6 - 180 + 3) + np.arctan2(np.sinh(eta_p), np.cos(xi_p))
    return np.degrees(lon), np.degrees(lat)

def make_geo(transform, crs):
    """Return f(col, row) -> (lon, lat) for continuous raster pixel coordinates (col,row = fractional pixel indices, pixel centre at +0.5)."""
    t = transform
    def to_xy(c, r): return t.a * c + t.b * r + t.c, t.d * c + t.e * r + t.f
    if crs is None: return None
    try:
        from pyproj import Transformer, CRS
        tr = Transformer.from_crs(CRS.from_wkt(crs.to_wkt()), 4326, always_xy=True)
        return lambda c, r: tr.transform(*to_xy(c, r))
    except Exception:
        s = crs.to_string() if hasattr(crs, 'to_string') else str(crs)
        import re
        m = re.search(r'EPSG:(32[67])(\d\d)', s)
        if m: return lambda c, r: _utm_inverse(*to_xy(c, r), zone=int(m.group(2)), north=(m.group(1) == '326'))
        print('WARNING: could not interpret the CRS; coordinate ticks are omitted. Install pyproj (pip install pyproj).'); return None

def _dm(t, kind):
    tot = int(round(abs(t) * 60)); d, m = divmod(tot, 60)
    h = ('E' if t >= 0 else 'W') if kind == 'lon' else ('N' if t >= 0 else 'S')
    return f'{d}°{m:02d}′{h}'

def coordinate_ticks(ax, geo, xlim, ylim, c0, r0, major, minor):
    """Ticks and labels on the LEFT (latitude) and BOTTOM (longitude) edges only; no grid lines; top and right edges are left clear."""
    ax.tick_params(axis='both', top=False, right=False, labeltop=False, labelright=False)
    if geo is None: ax.set_xticks([]); ax.set_yticks([]); return
    u0, u1 = xlim; vb, vt = ylim
    us = np.linspace(u0, u1, 900); lon, _ = geo(us + c0 + 0.5, np.full_like(us, vb) + r0 + 0.5)
    vs = np.linspace(vt, vb, 900); _, lat = geo(np.full_like(vs, u0) + c0 + 0.5, vs + r0 + 0.5)
    def pos(vals, coord, step_min):
        step = step_min / 60.0; lo, hi = vals.min(), vals.max()
        t = np.arange(np.ceil(lo / step - 1e-9), np.floor(hi / step + 1e-9) + 1) * step
        o = np.argsort(vals); return t, (np.interp(t, vals[o], coord[o]) if len(t) else np.array([]))
    for axis, vals, coord, kind in (('x', lon, us, 'lon'), ('y', lat, vs, 'lat')):
        tM, pM = pos(vals, coord, major); tm, pm = pos(vals, coord, minor)
        keep = np.array([abs(((x * 60) / major) - round((x * 60) / major)) > 1e-6 for x in tm], bool) if len(tm) else np.array([], bool)
        labs = [_dm(t, kind) for t in tM]
        if axis == 'x': ax.set_xticks(pM); ax.set_xticklabels(labs); ax.set_xticks(pm[keep], minor=True)
        else: ax.set_yticks(pM); ax.set_yticklabels(labs, rotation=90, va='center'); ax.set_yticks(pm[keep], minor=True)
    ax.tick_params(which='major', direction='out', length=TICK_MAJOR, width=0.8, labelsize=TICK_FS, pad=2.0)
    ax.tick_params(which='minor', direction='out', length=TICK_MINOR, width=0.6)

# ---------------- alternating scale bar and north arrow (your notebook's design), drawn inside the bottom band ----------------
def scale_bar(ax, s_in_px, PW, PH, prim_km, sec_km, nprim=4, x_in=0.10):
    """Bar of two secondary divisions (sec_km each) followed by nprim-1 primary divisions (prim_km each), black/white alternating;
    all labels in one row above the bar (0 5 10 20 30 40); unit 'km' to the right. Labels that would touch are thinned out."""
    L = s_in_px * 1000.0 / PIX; tr = ax.transAxes                       # inches per km
    def R(x0, w, y0, h, fc): ax.add_patch(Rectangle((x0 / PW, y0 / PH), w / PW, h / PH, transform=tr, fc=fc, ec='black', lw=1.2, zorder=6, clip_on=False))
    def T(x, y, s, **k): ax.text(x / PW, y / PH, s, transform=tr, fontsize=SCALEBAR_FS, ha='center', zorder=7, clip_on=False, **k)
    yP, hP = 0.08, 0.05
    widths = [sec_km, sec_km] + [prim_km] * (nprim - 1)
    pos, x = [0.0], x_in
    for i, w in enumerate(widths):
        R(x, w * L, yP, hP, 'black' if i % 2 == 0 else 'white'); x += w * L; pos.append(pos[-1] + w)
    fmt = lambda v: f'{v:g}'
    keep = [0]                                                           # drop a label when its neighbour would touch it; the last label is always kept
    for i in range(1, len(pos)):
        need = 0.5 * SCALEBAR_FS / 72 * (len(fmt(pos[i])) + len(fmt(pos[keep[-1]]))) / 2 + 0.01
        if (pos[i] - pos[keep[-1]]) * L >= need: keep.append(i)
        elif i == len(pos) - 1 and len(keep) > 1: keep[-1] = i
    for i in keep: T(x_in + pos[i] * L, yP + hP + 0.04, fmt(pos[i]), va='bottom')
    ax.text((x + 0.06) / PW, (yP + hP / 2) / PH, 'km', transform=tr, fontsize=SCALEBAR_FS, ha='left', va='center', zorder=7, clip_on=False)

def north_arrow(ax, PW, PH):
    x = (PW - 0.17) / PW
    ax.annotate('', xy=(x, 0.255 / PH), xytext=(x, 0.06 / PH), xycoords='axes fraction', zorder=7,
                arrowprops=dict(arrowstyle='-|>', lw=1.2, color='black', mutation_scale=7, shrinkA=0, shrinkB=0))
    ax.text(x, 0.275 / PH, 'N', transform=ax.transAxes, ha='center', va='bottom', fontsize=12.0, fontweight='bold', zorder=7)

# ---------------- frame builder ----------------
def draw_panel(fig, Wf, Hf, x_in, y_in, PW, PH, img_fn, nw, nh, c0, r0, geo, ticks, bar, arrow=False, title=None, title_fs=None):
    """One framed map panel. The map sits inside the frame with PADX/PADT white margin; the bottom band holds scale bar and north arrow."""
    ax = fig.add_axes([x_in / Wf, y_in / Hf, PW / Wf, PH / Hf]); ax.set_facecolor('white')
    img_fn(ax)
    for bc, br in BOUND: ax.plot(bc - c0, br - r0, color='black', lw=OUTLINE_LW, zorder=6, clip_on=True)   # study-area outline
    s = (PW - 2 * PADX) / nw                                              # inches per pixel
    xlim = (-0.5 - PADX / s, nw - 0.5 + PADX / s); ylim = (-0.5 + (PH - PADT) / s, -0.5 - PADT / s)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('auto')
    for sp in ax.spines.values(): sp.set_linewidth(0.8); sp.set_color('black')
    coordinate_ticks(ax, geo, xlim, (ylim[0], ylim[1]), c0, r0, *ticks)
    if bar: scale_bar(ax, s, PW, PH, *bar)
    if arrow: north_arrow(ax, PW, PH)
    if title:
        t = ax.set_title(title, loc='left', fontsize=title_fs or TITLE_FS, fontweight='bold', pad=8.0, linespacing=1.4); rend = fig.canvas.get_renderer()
        room = min(PW + 0.10, Wf - x_in - 0.03)                                 # never wider than the frame (+0.10 in) and never past the figure's right edge
        while t.get_window_extent(rend).width / fig.dpi > room and t.get_fontsize() > 6.5: t.set_fontsize(t.get_fontsize() - 0.25)   # shrink until the title fits above its frame
    return ax

def layout(nrows, ncols, map_h_in, title_lines, legend_h):
    """Absolute layout in inches. Returns (fig, Wf, Hf, PW, PH, xs, ys)."""
    Wf = W; PW = (Wf - RM - ncols * SEP) / ncols; PH = map_h_in + PADT + PADB
    TT = TT2 if title_lines == 2 else TT1
    Hf = TT + nrows * PH + (nrows - 1) * (TL + TT) + TL + legend_h
    fig = plt.figure(figsize=(Wf, Hf))
    xs = [SEP + j * (PW + SEP) for j in range(ncols)]
    ys = [Hf - TT - (i + 1) * PH - i * (TL + TT) for i in range(nrows)]
    return fig, Wf, Hf, PW, PH, xs, ys

def legend_at(fig, Hf, y_in, handles, ncol, title=None, fs=None):
    """Legend centred below the panels (your notebook's sizes); font and column spacing shrink only if it would touch the figure edge."""
    fs = fs or LEGEND_FS; tfs = LEGEND_TITLE_FS; rend = fig.canvas.get_renderer(); cs = 1.3
    while True:
        leg = fig.legend(handles=handles, loc='lower center', ncol=ncol, bbox_to_anchor=(0.5, y_in / Hf), handlelength=1.3, handleheight=1.0, columnspacing=cs,
                         frameon=False, fontsize=fs, title=title, title_fontsize=tfs, borderaxespad=1.0, labelspacing=0.5)
        if leg.get_window_extent(rend).width / fig.dpi <= W - 0.12 or fs <= 7: return leg
        leg.remove(); fs -= 0.25; tfs -= 0.25; cs = max(0.5, cs - 0.1)

# ---------------- map content ----------------
CMAP = ListedColormap(CLS_COL); CMAP.set_bad('white')
def class_img(a, win=None):
    """win = None -> whole frame window; else a square zoom window (row0, col0, n)."""
    w4 = fullwin() if win is None else (win[0], win[1], win[2], win[2])
    b = crop(a, w4); m = (b < 0); om = outside_mask(w4)
    if om is not None: m = m | om
    b = np.ma.masked_where(m, b); return b, (lambda ax, b=b: ax.imshow(b, cmap=CMAP, vmin=0, vmax=K - 1, interpolation='nearest'))
CLASS_HANDLES = [Patch(fc=c, ec='#444', lw=0.4, label=n) for c, n in zip(CLS_COL, CLASS_NAMES)]

def full_s(PW): return (PW - 2 * PADX) / fullwin()[3]             # inches per pixel for a whole-area panel
def full_h(PW): return fullwin()[2] * full_s(PW)

def fig3(L):
    n = 3; PW = (W - RM - n * SEP) / n; fr0, fc0, fh, fw = fullwin()
    fig, Wf, Hf, PW, PH, xs, ys = layout(1, n, full_h(PW), 1, 0.40)
    for j, (y, t) in enumerate(zip(['2003', '2013', '2023'], [TT('a', '2003'), TT('b', '2013'), TT('c', '2023')])):
        b, fn = class_img(L[y]); draw_panel(fig, Wf, Hf, xs[j], ys[0], PW, PH, fn, fw, fh, fc0, fr0, GEO, FULL_TICKS, BAR_FULL, arrow=(j == n - 1), title=t)
    legend_at(fig, Hf, 0.03, CLASS_HANDLES, 5); savefig(fig, 'Fig3_LULC_2003_2013_2023')

def busiest_window(true, prev, valid, n=520, stride=40):
    ch = ((true != prev) & valid)
    if ZOOM_IGNORE_WATER: ch = ch & (true != 0) & (prev != 0)            # class index 0 = Water
    ch = ch.astype(np.int32)
    if INSIDE is not None: ch = ch * INSIDE
    I = ch.cumsum(0).cumsum(1); I = np.pad(I, ((1, 0), (1, 0)))
    best, bpos = -1, (0, 0)
    for r in range(0, true.shape[0] - n + 1, stride):
        for c in range(0, true.shape[1] - n + 1, stride):
            s = I[r + n, c + n] - I[r, c + n] - I[r + n, c] + I[r, c]
            if s > best: best, bpos = s, (r, c)
    return bpos[0], bpos[1], n

AG_COL = {'hit': '#009E73', 'miss': '#CC79A7', 'wrong': '#D55E00', 'fa': '#F0E442', 'ok': '#EDEDED'}
def agreement(true, pred, prev):
    ch = true != prev; out = np.full(true.shape, np.nan)
    out[~ch & (pred == prev)] = 0; out[ch & (pred == true)] = 1; out[ch & (pred == prev)] = 2; out[ch & (pred != prev) & (pred != true)] = 3; out[~ch & (pred != prev)] = 4
    out[~VALID] = np.nan
    if INSIDE is not None: out[~INSIDE] = np.nan
    return out
AG_CMAP = ListedColormap([AG_COL['ok'], AG_COL['hit'], AG_COL['miss'], AG_COL['wrong'], AG_COL['fa']]); AG_CMAP.set_bad('white')

def fig4(L, HIND, win=None):
    win = win or busiest_window(L['2023'], L['2013'], VALID); r0, c0, n = win; fr0, fc0, fh, fw = fullwin(); ZOOM_KM[0] = n * PIX / 1000
    n_c = 3; PW = (W - RM - n_c * SEP) / n_c; s_zoom = (PW - 2 * PADX) / n
    fig, Wf, Hf, PW, PH, xs, ys = layout(2, n_c, max(full_h(PW), n * s_zoom), 1, 1.28)
    def P(i, j, fn, nw, nh, cc, rr, bar, title, arrow=False, zoom=True):
        draw_panel(fig, Wf, Hf, xs[j], ys[i], PW, PH, fn, nw, nh, cc, rr, GEO, ZOOM_TICKS if zoom else FULL_TICKS, bar, arrow=arrow, title=title)
    def full_fn(ax):
        b, fn = class_img(L['2023']); fn(ax); ax.add_patch(Rectangle((c0 - fc0 - 0.5, r0 - fr0 - 0.5), n, n, fill=False, ec='red', lw=1.2, ls='--', zorder=8))
    P(0, 0, full_fn, fw, fh, fc0, fr0, BAR_FULL, TT('a', 'Observed 2023'), zoom=False)
    b, fn = class_img(L['2013'], win); P(0, 1, fn, n, n, c0, r0, BAR_ZOOM, TT('b', 'Persistence (2013)'))
    b, fn = class_img(L['2023'], win); P(0, 2, fn, n, n, c0, r0, BAR_ZOOM, TT('c', 'Observed 2023 (zoom)'), arrow=True)
    for j, (m, t) in enumerate(zip(['CA_ANN_Markov', 'CNN_Markov', 'CNN_argmax'], [TT('d', 'CA-ANN-Markov'), TT('e', 'CNN-Markov'), TT('f', 'CNN-argmax')])):
        ag = np.ma.masked_invalid(agreement(L['2023'], HIND[m], L['2013'])[r0:r0 + n, c0:c0 + n])
        P(1, j, (lambda ax, ag=ag: ax.imshow(ag, cmap=AG_CMAP, vmin=0, vmax=4, interpolation='nearest')), n, n, c0, r0, BAR_ZOOM, t, arrow=(j == 2))
    ag_h = [Patch(fc=AG_COL[k], ec='#444', lw=0.4, label=l) for k, l in [('hit', 'Hit'), ('miss', 'Miss'), ('wrong', 'Wrong hit'), ('fa', 'False alarm'), ('ok', 'Persistent, correct')]]
    legend_at(fig, Hf, 0.03, ag_h, 5, title='Agreement with observed 2023 (panels d\u2013f)')
    legend_at(fig, Hf, 0.58, CLASS_HANDLES, 5, title='Land cover (panels a\u2013c)')
    savefig(fig, 'Fig4_hindcast_maps')

def change_rgb(fc, base):
    """Changed pixels in their 2033 class colour on light grey. Changed pixels are enlarged to 5 x 5 px so they survive down-sampling to print size."""
    from scipy.ndimage import maximum_filter
    rgb = np.ones(base.shape + (3,)); rgb[VALID] = mpl.colors.to_rgb(AG_COL['ok'])
    ch = (fc != base) & VALID
    for k in range(K):
        m = maximum_filter(ch & (fc == k), size=5) & VALID; rgb[m] = mpl.colors.to_rgb(CLS_COL[k])
    if INSIDE is not None: rgb[~INSIDE] = 1.0                             # display only: pixels outside the study-area polygon are shown white
    return rgb, 100 * ch.sum() / VALID.sum()                              # percentage of the full analysis domain (same denominator as Table 7)

def fig10(L, FORE):
    n_c = 3; PW = (W - RM - n_c * SEP) / n_c; fr0, fc0, fh, fw = fullwin()
    fig, Wf, Hf, PW, PH, xs, ys = layout(2, n_c, full_h(PW), 2 if PANEL_TITLES == 'full' else 1, 0.40)
    lab = {'T2': 'T2 rates (2013\u201323)', 'Mean': 'Mean rates', 'T1': 'T1 rates (2003\u201313)'}
    for j, sc in enumerate(['T2', 'Mean', 'T1']):
        b, fn = class_img(FORE[sc]); draw_panel(fig, Wf, Hf, xs[j], ys[0], PW, PH, fn, fw, fh, fc0, fr0, GEO, FULL_TICKS, BAR_FULL, arrow=(j == 2), title=TT('abc'[j], f'2033 projection\n{lab[sc]}'))
        rgb, pc = change_rgb(FORE[sc], L['2023']); rgb = crop(rgb, fullwin()); pc = globals().get('SEED_CHG', {}).get(sc, pc)   # per-seed mean (matches Table 7); falls back to the vote map
        draw_panel(fig, Wf, Hf, xs[j], ys[1], PW, PH, (lambda ax, rgb=rgb: ax.imshow(rgb, interpolation='nearest')), fw, fh, fc0, fr0, GEO, FULL_TICKS, BAR_FULL, arrow=(j == 2),
                   title=TT('def'[j], f'Change from 2023\n{pc:.2f}% of pixels'))
        PCT_LOG[sc] = pc
    legend_at(fig, Hf, 0.03, CLASS_HANDLES + [Patch(fc=AG_COL['ok'], ec='#444', lw=0.4, label='Unchanged')], 6)
    savefig(fig, 'Fig10_forecast_maps_2033')

def print_caption_text():
    """Caption wording that replaces the removed panel titles (numbers are taken from this run)."""
    print('\n' + '=' * 78 + '\nSuggested figure captions (panel titles were reduced to letters)\n' + '=' * 78)
    print('Fig. 3. Land use and land cover of the Khatra sub-division in (a) 2003, (b) 2013 and (c) 2023.\n')
    if ZOOM_KM[0]:
        print(f'Fig. 4. Out-of-time hindcast of 2023 from the 2013 map. (a) Observed 2023 land cover for the whole sub-division; the red dashed box marks the {ZOOM_KM[0]:.1f} km square '
              'window with the most observed 2013-2023 change that is enlarged in the other panels. (b) Persistence (no-change) forecast, i.e. the 2013 map, and (c) observed 2023 in the window. '
              '(d) CA-ANN-Markov, (e) CNN-Markov and (f) CNN-argmax, shown as agreement with observed 2023 (hit, miss, wrong hit, false alarm, persistent and correct).\n')
    if PCT_LOG:
        p = lambda k: f"{PCT_LOG[k]:.2f}%" if k in PCT_LOG else 'n/a'
        print('Fig. 10. Projected land use and land cover for 2033 from CNN-Markov under (a) T2 (2013-23), (b) mean and (c) T1 (2003-13) Markov rates, with (d)-(f) the corresponding changes relative to 2023. '
              f'Changed pixels are drawn in their 2033 class colour on light grey and enlarged to 5 x 5 pixels for legibility; they cover {p("T2")}, {p("Mean")} and {p("T1")} of the study area in (d), (e) and (f). '
              'Maps show the five-seed majority vote; percentages are seed means.')

GEO = make_geo(_TF, _CRS)
_bp = find_boundary(PATH)
try:
    if not _bp: raise FileNotFoundError('no boundary file found')
    print('using boundary:', _bp); _in, _ln = load_boundary(_bp, _TF, _CRS, VALID.shape); set_boundary(_in, _ln, VALID.shape)
    if _in is not None: print(f'valid pixels outside the polygon: {(VALID & ~_in).sum():,} (about 0 if the shapefile is the footprint of your LULC rasters)')
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'geopandas'], check=False); print('geopandas was missing and has been installed - restart the runtime and run again to draw the outline.'); set_boundary(None, [], VALID.shape)
except Exception as e:
    print('Boundary not drawn:', e, '- set BOUNDARY_PATH to the .shp file.'); set_boundary(None, [], VALID.shape)
if 'LULC' in globals() and len(LULC) == 3:
    fig3(LULC)
    if all(v is not None for v in HIND.values()): fig4(LULC, HIND)
    if all(v is not None for v in FORE.values()): fig10(LULC, FORE)
    print_caption_text()

### Notes: what differs from the edited v2 notebook
* **Frame limits corrected.** v2 had `-0.5 * PADX / s` and `-0.5 * PADT / s` (a `*` instead of `-`). That halved the left and top white margins and made every scale bar about 2 % too short (a bar labelled 40 km spanned about 39.2 km). Margins are now equal on all sides and the bar length matches the map.
* **Scale bar.** Same design as v2 (5 + 5 + 10 + 10 + 10 km, labels 0 5 10 20 30 40 in one row above the bar), but the numbers are 8.5 pt instead of 11.5 pt so that "5" and "10" (only about 0.11 in apart) no longer touch, and the unit is "km" (SI), not "Km". Labels that would still touch are dropped automatically. Change `SCALEBAR_FS` if you prefer.
* **Text that would overflow.** Panel titles and legends keep your sizes (13 pt titles, 11.5 pt legends) and shrink only if they would run past the frame or the figure edge (v2 let the Fig. 4 panel (c) title and the Fig. 10 "Unchanged" label spill out, which made the saved PNGs wider than 6.3 in).
* **Boundary shapefile** is read once (v2 re-read it for each of the 15 panels) and the outline is aligned to pixel centres (v2 was half a pixel off). Pixels outside the polygon are not repainted unless `MASK_OUTSIDE = True`; the frame is not cropped unless `CROP_TO_BOUNDARY = True`.
* **Fig. 10 percentages** in the lower panel titles are the mean over the five seeds of the share of valid pixels that change (the Table 7 numbers). The map itself is the five-seed majority vote, which shows fewer changed pixels. Caption wording: "Maps show the five-seed majority vote; percentages are seed means."
* **Fig. 10 lower row.** Changed pixels are drawn enlarged to 5 x 5 pixels in their 2033 class colour on light grey, otherwise they vanish when about 2,400 pixels are reduced to about 1.7 inches. State this in the caption.
* **Fig. 4 zoom window** is chosen automatically as the 520 x 520-pixel window (about 15.6 km) with the most observed 2013-2023 change, which currently lands on the reservoir. Set `ZOOM_IGNORE_WATER = True` to choose by non-water change, or pass `fig4(LULC, HIND, win=(row0, col0, 520))`.
* **Coordinates.** Latitude labels (left) and longitude labels (bottom) come from the GeoTIFF's CRS; no grid lines; top and right edges clear. Whole-area panels: labels every 15 arc-minutes, minor ticks every 5. The zoom window is only about 9 arc-minutes wide, so its panels use 5-minute labels and 1-minute minor ticks (`ZOOM_TICKS`).
* **Do not stretch** the figures in Word. For EPS: `pdftocairo -eps Fig3_LULC_2003_2013_2023.pdf Fig3.eps`.